In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
app_path = "/content/drive/MyDrive/app.py"

with open(app_path, "r") as file:
    app_code = file.read()

print(app_code)


import streamlit as st
import pandas as pd
import numpy as np
import xgboost as xgb
import tensorflow as tf
import os

from PIL import Image
from tensorflow.keras.applications.resnet50 import preprocess_input

# --------------------------------------------------
# GROQ GENAI CONFIGURATION
# --------------------------------------------------
from groq import Groq

GROQ_MODEL = "llama-3.3-70b-versatile"

ADVISORY_DISCLAIMER = (
    "This AI-generated information is for educational purposes only. "
    "It is not a medical diagnosis or treatment recommendation. "
    "Please consult a qualified healthcare professional."
)

def get_groq_client():
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
    except Exception:
        key = os.environ.get("GROQ_API_KEY")

    if not key:
        key = os.environ.get("GROQ_API_KEY")

    return Groq(api_key=key) if key else None


# --------------------------------------------------
# PAGE CONFIGURATION
# 

In [3]:
!pip install -q streamlit xgboost tensorflow shap groq pillow pandas numpy pyngrok

In [4]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if GROQ_API_KEY:
    print("Groq API key loaded successfully!")
else:
    print("Groq API key not found. Check Colab Secrets.")

Groq API key loaded successfully!


In [5]:
app_path = "/content/drive/MyDrive/app.py"

with open(app_path, "r") as file:
    app_code = file.read()

print("app.py loaded successfully!")
print("Total lines:", len(app_code.splitlines()))

app.py loaded successfully!
Total lines: 522


In [6]:
for i, line in enumerate(app_code.splitlines(), start=1):
    if "st.sidebar.radio" in line or 'page ==' in line:
        print(f"{i}: {line}")

115: page = st.sidebar.radio(
129: if page == "Home":
159: elif page == "Clinical Analysis":
301: elif page == "SHAP Explainability":
434: elif page == "Ultrasound Analysis":


In [7]:
app_code = app_code.replace(
    '"SHAP Explainability",\n',
    ''
)

print("Updated sidebar navigation.")

Updated sidebar navigation.


In [8]:
for i, line in enumerate(app_code.splitlines(), start=1):
    if "st.sidebar.radio" in line or "SHAP Explainability" in line:
        print(f"{i}: {line}")

115: page = st.sidebar.radio(
142:     - SHAP Explainability: Understand the influence of
300: elif page == "SHAP Explainability":
302:     st.title("SHAP Explainability")


In [9]:
start = app_code.index('elif page == "SHAP Explainability":')
end = app_code.index('elif page == "Ultrasound Analysis":')

app_code = app_code[:start] + app_code[end:]

print("Separate SHAP page removed successfully!")

Separate SHAP page removed successfully!


In [10]:
app_code = app_code.replace(
    "- SHAP Explainability: Understand the influence of",
    "- Patient-specific SHAP: Understand the features influencing the clinical prediction."
)

print("Home page description updated.")

Home page description updated.


In [11]:
for i, line in enumerate(app_code.splitlines(), start=1):
    if "GROQ_MODEL" in line or "get_groq_client" in line or "ADVISORY_DISCLAIMER" in line:
        print(f"{i}: {line}")

17: GROQ_MODEL = "llama-3.3-70b-versatile"
19: ADVISORY_DISCLAIMER = (
25: def get_groq_client():


In [12]:
lines = app_code.splitlines()

for i in range(0, 45):
    print(f"{i+1}: {lines[i]}")

1: 
2: import streamlit as st
3: import pandas as pd
4: import numpy as np
5: import xgboost as xgb
6: import tensorflow as tf
7: import os
8: 
9: from PIL import Image
10: from tensorflow.keras.applications.resnet50 import preprocess_input
11: 
12: # --------------------------------------------------
13: # GROQ GENAI CONFIGURATION
14: # --------------------------------------------------
15: from groq import Groq
16: 
17: GROQ_MODEL = "llama-3.3-70b-versatile"
18: 
19: ADVISORY_DISCLAIMER = (
20:     "This AI-generated information is for educational purposes only. "
21:     "It is not a medical diagnosis or treatment recommendation. "
22:     "Please consult a qualified healthcare professional."
23: )
24: 
25: def get_groq_client():
26:     try:
27:         from google.colab import userdata
28:         key = userdata.get("GROQ_API_KEY")
29:     except Exception:
30:         key = os.environ.get("GROQ_API_KEY")
31: 
32:     if not key:
33:         key = os.environ.get("GROQ_API_KEY")
34

In [13]:
new_groq_function = '''def get_groq_client():
    from google.colab import userdata

    key = userdata.get("GROQ_API_KEY")

    if not key:
        return None

    return Groq(api_key=key)
'''

start = app_code.index("def get_groq_client():")
end = app_code.index("\\n\\n", app_code.index("return Groq(api_key=key) if key else None", start))

app_code = app_code[:start] + new_groq_function.rstrip() + app_code[end:]

print("Groq client configuration updated!")

ValueError: substring not found

In [14]:
start = app_code.index("def get_groq_client():")
end = app_code.index("# --------------------------------------------------", start)

new_groq_function = '''def get_groq_client():
    from google.colab import userdata

    key = userdata.get("GROQ_API_KEY")

    if not key:
        return None

    return Groq(api_key=key)


'''

app_code = (
    app_code[:start]
    + new_groq_function
    + app_code[end:]
)

print("Groq client configuration updated successfully!")

Groq client configuration updated successfully!


In [15]:
advisory_functions = '''

# --------------------------------------------------
# CLINICAL AI ADVISORY
# --------------------------------------------------

def generate_clinical_advisory(subtype, confidence, patient_features, shap_features):

    client = get_groq_client()

    if client is None:
        return "Groq API key not found. Please check Colab Secrets."

    prompt = f"""
    You are an educational PCOS health information assistant.

    Clinical model predicted subtype: {subtype}
    Prediction confidence: {confidence:.2f}

    Patient input features:
    {patient_features}

    Top SHAP contributing features:
    {shap_features}

    Explain the model prediction cautiously.
    SHAP values indicate model contributions, not medical causes.

    Provide:
    1. Simple explanation of the model result.
    2. General dietary guidance.
    3. Physical activity suggestions.
    4. Sleep and stress management suggestions.
    5. Tests or follow-up topics to discuss with a doctor.

    Do not diagnose, prescribe medicines or supplements, suggest doses,
    or infer unmeasured hormone levels.

    Keep the response clear and concise.
    """

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return response.choices[0].message.content


# --------------------------------------------------
# ULTRASOUND AI ADVISORY
# --------------------------------------------------

def generate_ultrasound_advisory(prediction, pcos_probability,
                                 comparison_probability):

    client = get_groq_client()

    if client is None:
        return "Groq API key not found. Please check Colab Secrets."

    prompt = f"""
    You are an educational PCOS health information assistant.

    CNN ultrasound classification: {prediction}
    PCO probability: {pcos_probability:.2%}
    Comparison Class probability: {comparison_probability:.2%}

    Explain the image classification in simple language.

    Provide:
    1. Explanation of the CNN result.
    2. General information about ultrasound findings.
    3. Limitations of AI-based ultrasound classification.
    4. Questions to discuss with a qualified healthcare professional.

    Do not diagnose PCOS or infer clinical subtypes or hormone levels.
    Do not prescribe medicines, supplements or doses.

    Keep the response clear and concise.
    """

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return response.choices[0].message.content

'''

app_code = app_code.replace(
    "# --------------------------------------------------\n# PAGE CONFIGURATION",
    advisory_functions + "\n# --------------------------------------------------\n# PAGE CONFIGURATION"
)

print("Both AI advisory functions added successfully!")

Both AI advisory functions added successfully!


In [16]:
lines = app_code.splitlines()

start = next(
    i for i, line in enumerate(lines)
    if 'elif page == "Clinical Analysis":' in line
)

end = next(
    i for i, line in enumerate(lines)
    if 'elif page == "Ultrasound Analysis":' in line
)

for i in range(start, end):
    print(f"{i+1}: {lines[i]}")

252: elif page == "Clinical Analysis":
253: 
254:     st.title("Clinical Analysis")
255: 
256:     st.write("Enter the clinical measurements below.")
257: 
258:     with st.form("clinical_form"):
259: 
260:         col1, col2 = st.columns(2)
261: 
262:         with col1:
263: 
264:             hair = st.selectbox("Hair Growth", ["No", "Yes"])
265:             blood_sugar = st.number_input("Blood Sugar", min_value=0.0, value=100.0)
266:             hormonal = st.number_input("Hormonal Disruption", min_value=0.0, value=1.0)
267:             amh = st.number_input("AMH", min_value=0.0, value=3.0)
268:             androgen = st.number_input("Androgen Score", min_value=0.0, value=1.0)
269:             homa = st.number_input("HOMA_IR", min_value=0.0, value=1.0)
270:             fsh_lh_raw = st.number_input("FSH_LH_raw", min_value=0.0, value=1.0)
271:             total_follicles = st.number_input("Total Follicles", min_value=0.0, value=20.0)
272:             follicle_l = st.number_input("Folli

In [17]:
clinical_button_code = '''
            st.divider()
            st.subheader("Clinical AI Advisory")

            if st.button("Generate Clinical AI Advisory"):
                with st.spinner("Generating clinical advisory..."):
                    try:
                        advisory = generate_clinical_advisory(
                            subtype=CLASSES[predicted_index],
                            confidence=float(probabilities[predicted_index]),
                            patient_features=values,
                            shap_features=shap_df[
                                ["Feature", "SHAP Value"]
                            ].to_dict(orient="records")
                        )

                        st.markdown(advisory)
                        st.info(ADVISORY_DISCLAIMER)

                    except Exception as e:
                        st.error(f"Clinical advisory error: {e}")
'''

anchor = '''            st.dataframe(
                probability_df[["Subtype", "Probability (%)"]],
                use_container_width=True
            )
'''

if anchor not in app_code:
    print("Insertion point not found. No changes made.")
else:
    app_code = app_code.replace(
        anchor,
        anchor + clinical_button_code,
        1
    )
    print("Clinical AI Advisory button added successfully!")

Clinical AI Advisory button added successfully!


In [18]:
lines = app_code.splitlines()

start = next(
    i for i, line in enumerate(lines)
    if 'elif page == "Ultrasound Analysis":' in line
)

for i in range(start, len(lines)):
    print(f"{i+1}: {lines[i]}")

415: elif page == "Ultrasound Analysis":
416: 
417:     st.title("Ultrasound Analysis")
418: 
419:     st.write("Upload an ultrasound image for model-based classification.")
420: 
421:     uploaded_file = st.file_uploader(
422:         "Upload ultrasound image",
423:         type=["jpg", "jpeg", "png"]
424:     )
425: 
426:     if uploaded_file is not None:
427: 
428:         image = Image.open(uploaded_file).convert("L")
429: 
430:         st.image(
431:             image,
432:             caption="Uploaded Ultrasound Image",
433:             width=350
434:         )
435: 
436:         if st.button("Analyze Ultrasound"):
437: 
438:             try:
439: 
440:                 model = load_ultrasound_model()
441: 
442:                 resized_image = image.resize(
443:                     (224, 224),
444:                     Image.Resampling.BICUBIC
445:                 )
446: 
447:                 image_array = np.array(
448:                     resized_image.convert("RGB"),
449:      

In [19]:
ultrasound_button_code = '''
                st.divider()
                st.subheader("Ultrasound AI Advisory")

                if st.button("Generate Ultrasound AI Advisory"):
                    with st.spinner("Generating ultrasound advisory..."):
                        try:
                            advisory = generate_ultrasound_advisory(
                                prediction=prediction,
                                pcos_probability=pcos_probability,
                                comparison_probability=comparison_probability
                            )

                            st.markdown(advisory)
                            st.info(ADVISORY_DISCLAIMER)

                        except Exception as e:
                            st.error(f"Ultrasound advisory error: {e}")
'''

anchor = '''                st.dataframe(
                    probability_df,
                    use_container_width=True
                )
'''

if anchor not in app_code:
    print("Insertion point not found. No changes made.")
else:
    app_code = app_code.replace(
        anchor,
        anchor + ultrasound_button_code,
        1
    )
    print("Ultrasound AI Advisory button added successfully!")

Ultrasound AI Advisory button added successfully!


In [20]:
app_code = app_code.replace(
    'CLINICAL_MODEL_PATH = "/content/drive/MyDrive/xgboost_model.json"',
    'CLINICAL_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"'
)

app_code = app_code.replace(
    'ULTRASOUND_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras"',
    'ULTRASOUND_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras"'
)

print("Model paths updated.")

Model paths updated.


In [21]:
for line in app_code.splitlines():
    if "MODEL_PATH" in line:
        print(line)

CLINICAL_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"
ULTRASOUND_MODEL_PATH = (
    model.load_model(CLINICAL_MODEL_PATH)
    return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)


In [22]:
for i, line in enumerate(app_code.splitlines(), 1):
    if "CLINICAL_MODEL_PATH" in line or "ULTRASOUND_MODEL_PATH" in line:
        print(f"{i}: {line}")

146: CLINICAL_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"
148: ULTRASOUND_MODEL_PATH = (
180:     model.load_model(CLINICAL_MODEL_PATH)
186:     return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)


In [23]:
for i, line in enumerate(app_code.splitlines(), 1):
    if "def load_clinical_model" in line or "def load_ultrasound_model" in line:
        print(f"\nFunction found at line {i}")
        for j in range(i - 1, min(i + 10, len(app_code.splitlines()))):
            print(f"{j + 1}: {app_code.splitlines()[j]}")


Function found at line 178
178: def load_clinical_model():
179:     model = xgb.Booster()
180:     model.load_model(CLINICAL_MODEL_PATH)
181:     return model
182: 
183: 
184: @st.cache_resource
185: def load_ultrasound_model():
186:     return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)
187: 
188: 

Function found at line 185
185: def load_ultrasound_model():
186:     return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)
187: 
188: 
189: # --------------------------------------------------
190: # LOAD SHAP CSV FILES
191: # --------------------------------------------------
192: 
193: @st.cache_data
194: def load_shap_data():
195: 


In [24]:
lines = app_code.splitlines()

for i in range(140, 210):
    print(f"{i+1}: {lines[i]}")

141: 
142: # --------------------------------------------------
143: # MODEL AND DATA PATHS
144: # --------------------------------------------------
145: 
146: CLINICAL_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"
147: 
148: ULTRASOUND_MODEL_PATH = (
149:     "/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras"
150: )
151: 
152: SHAP_FOLDER = "/content/drive/MyDrive/pcos_project/outputs/csv"
153: 
154: TOP5_PATH = os.path.join(SHAP_FOLDER, "shap_top5_per_subtype.csv")
155: GLOBAL_PATH = os.path.join(SHAP_FOLDER, "shap_global_importance.csv")
156: FEATURE_PATH = os.path.join(SHAP_FOLDER, "shap_feature_importance.csv")
157: 
158: FEATURES = [
159:     "Hair_Growth", "Blood_Sugar", "Hormonal_Disruption", "AMH",
160:     "Androgen_Score", "HOMA_IR", "FSH_LH_raw", "Total_Follicles",
161:     "Follicle_L", "LH", "FSH", "LH_FSH_Ratio", "Cycle_Length",
162:     "Fast_Food", "TSH", "BP_Diastolic", "BMI", "Progesterone",
163:     "Marriage

In [25]:
for term in [
    "load_shap_data",
    "TOP5_PATH",
    "GLOBAL_PATH",
    "FEATURE_PATH",
    "SHAP_FOLDER"
]:
    print(f"\n{term}: {app_code.count(term)} occurrence(s)")


load_shap_data: 1 occurrence(s)

TOP5_PATH: 2 occurrence(s)

GLOBAL_PATH: 2 occurrence(s)

FEATURE_PATH: 2 occurrence(s)

SHAP_FOLDER: 4 occurrence(s)


In [26]:
start = app_code.index("# MODEL AND DATA PATHS")
end = app_code.index("FEATURES = [", start)

new_paths = '''# MODEL AND DATA PATHS
# --------------------------------------------------

CLINICAL_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"

ULTRASOUND_MODEL_PATH = (
    "/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras"
)

'''

app_code = app_code[:start] + new_paths + app_code[end:]

start = app_code.index("# LOAD SHAP CSV FILES")
end = app_code.index("# SIDEBAR NAVIGATION", start)

app_code = app_code[:start] + app_code[end:]

print("Old SHAP CSV paths and loader removed successfully!")

Old SHAP CSV paths and loader removed successfully!


In [27]:
for term in [
    "load_shap_data",
    "TOP5_PATH",
    "GLOBAL_PATH",
    "FEATURE_PATH",
    "SHAP_FOLDER",
    "TreeExplainer",
    "patient_shap",
    "Clinical AI Advisory",
    "Ultrasound AI Advisory"
]:
    print(f"{term}: {app_code.count(term)} occurrence(s)")

load_shap_data: 0 occurrence(s)
TOP5_PATH: 0 occurrence(s)
GLOBAL_PATH: 0 occurrence(s)
FEATURE_PATH: 0 occurrence(s)
SHAP_FOLDER: 0 occurrence(s)
TreeExplainer: 1 occurrence(s)
patient_shap: 2 occurrence(s)
Clinical AI Advisory: 2 occurrence(s)
Ultrasound AI Advisory: 2 occurrence(s)


In [28]:
import ast

try:
    ast.parse(app_code)
    print("Python syntax is valid!")
except SyntaxError as e:
    print("Syntax error:", e)
    print("Line:", e.lineno)

Python syntax is valid!


In [29]:
with open(app_path, "w") as file:
    file.write(app_code)

print("Updated app.py saved successfully to Google Drive!")

Updated app.py saved successfully to Google Drive!


In [30]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key loaded into the app environment!")

Groq API key loaded into the app environment!


In [31]:
start = app_code.index("def get_groq_client():")
end = app_code.index("# --------------------------------------------------", start)

new_groq_function = '''def get_groq_client():
    key = os.environ.get("GROQ_API_KEY")

    if not key:
        return None

    return Groq(api_key=key)


'''

app_code = app_code[:start] + new_groq_function + app_code[end:]

print("Groq client updated successfully!")

Groq client updated successfully!


In [32]:
with open(app_path, "w") as file:
    file.write(app_code)

print("Final app.py saved successfully!")

Final app.py saved successfully!


In [33]:
import ast

ast.parse(app_code)
print("Python syntax is valid!")

Python syntax is valid!


In [49]:
# Install Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import time
import re

# Start Streamlit
streamlit = subprocess.Popen(
    [
        "streamlit", "run",
        "/content/drive/MyDrive/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Start Cloudflare Tunnel
cloudflare = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Find the public URL
print("Starting PCOS Insight...")

for line in cloudflare.stdout:
    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
    if match:
        print("\nPCOS Insight is ready!")
        print("Open this URL:", match.group(0))
        break

(Reading database ... 122817 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) over (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Starting PCOS Insight...

PCOS Insight is ready!
Open this URL: https://likes-nathan-col-sporting.trycloudflare.com


In [35]:
import os

base_path = "/content/drive/MyDrive"

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith((".json", ".ubj", ".model")) and "xgboost" in file.lower():
            print(os.path.join(root, file))

In [36]:
!find /content/drive/MyDrive -iname "*.json" -o -iname "*.model" -o -iname "*.ubj"

In [37]:
import os

model_folder = "/content/drive/MyDrive/pcos_project/outputs/models"

print("Folder exists:", os.path.exists(model_folder))

if os.path.exists(model_folder):
    print("Files inside models folder:")
    print(os.listdir(model_folder))
else:
    print("Models folder not found!")

Folder exists: True
Files inside models folder:
['xgboost_model.pkl', 'nn_model.keras', 'cnn_model.keras']


In [38]:
import os

model_folder = "/content/drive/MyDrive/pcos_project/outputs/models"

print(os.listdir(model_folder))

['xgboost_model.pkl', 'nn_model.keras', 'cnn_model.keras', 'xgboost_model.json']


In [40]:
import os

print(os.path.exists(
    "/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"
))

True


In [41]:
!pkill -f "streamlit run" || true

^C


In [43]:
lines = app_code.splitlines()

for i in range(300, 415):
    print(f"{i+1}: {lines[i]}")

301: 
302:             probabilities = probabilities[0]
303: 
304:             predicted_index = int(np.argmax(probabilities))
305: 
306: 
307:             # Patient-specific SHAP explanation
308:             import shap
309: 
310:             explainer = shap.TreeExplainer(model)
311:             shap_values = np.asarray(explainer.shap_values(input_df))
312: 
313:             patient_shap = shap_values[0, :, predicted_index]
314: 
315:             shap_df = pd.DataFrame({
316:                 "Feature": FEATURES,
317:                 "SHAP Value": patient_shap
318:             })
319: 
320:             shap_df["Absolute SHAP"] = np.abs(shap_df["SHAP Value"])
321:             shap_df = shap_df.sort_values(
322:                 "Absolute SHAP", ascending=False
323:             ).head(5)
324: 
325:             st.subheader("Patient-Specific SHAP Explanation")
326:             st.caption("Top five features contributing to this prediction.")
327: 
328:             st.bar_chart(shap_df.set_

In [44]:
start = app_code.index("            # Patient-specific SHAP explanation")

end = app_code.index(
    "        except Exception as e:\n\n            st.error(f\"Clinical prediction error: {e}\")",
    start
)

new_clinical_output = '''
            # 1. Display predicted subtype first
            st.success(
                f"Predicted PCOS Subtype: {CLASSES[predicted_index]}"
            )

            st.metric(
                "Prediction Confidence",
                f"{probabilities[predicted_index] * 100:.2f}%"
            )

            # 2. Probability distribution
            probability_df = pd.DataFrame({
                "Subtype": CLASSES,
                "Probability": probabilities
            })

            probability_df["Probability (%)"] = (
                probability_df["Probability"] * 100
            ).round(2)

            st.subheader("Subtype Probability Distribution")

            st.bar_chart(
                probability_df.set_index("Subtype")["Probability (%)"]
            )

            st.dataframe(
                probability_df[["Subtype", "Probability (%)"]],
                use_container_width=True,
                hide_index=True
            )

            # 3. Patient-specific SHAP explanation
            import shap

            explainer = shap.TreeExplainer(model)
            shap_values = np.asarray(explainer.shap_values(input_df))

            patient_shap = shap_values[0, :, predicted_index]

            shap_df = pd.DataFrame({
                "Feature": FEATURES,
                "SHAP Value": patient_shap
            })

            shap_df["Absolute SHAP"] = np.abs(shap_df["SHAP Value"])
            shap_df = shap_df.sort_values(
                "Absolute SHAP", ascending=False
            ).head(5)

            st.subheader("Patient-Specific SHAP Explanation")
            st.caption(
                "Top five features contributing to this model prediction. "
                "SHAP values represent model contributions, not medical causes."
            )

            st.bar_chart(shap_df.set_index("Feature")["SHAP Value"])

            st.dataframe(
                shap_df[["Feature", "SHAP Value"]],
                hide_index=True
            )

            # 4. Automatically generate clinical advisory
            st.divider()
            st.subheader("Clinical AI Advisory")

            with st.spinner("Generating your clinical advisory..."):
                try:
                    advisory = generate_clinical_advisory(
                        subtype=CLASSES[predicted_index],
                        confidence=float(probabilities[predicted_index]),
                        patient_features=values,
                        shap_features=shap_df[
                            ["Feature", "SHAP Value"]
                        ].to_dict(orient="records")
                    )

                    st.markdown(advisory)
                    st.info(ADVISORY_DISCLAIMER)

                except Exception as e:
                    st.error(f"Clinical advisory error: {e}")

'''

app_code = app_code[:start] + new_clinical_output + app_code[end:]

print("Clinical results reordered and automatic advisory enabled!")

Clinical results reordered and automatic advisory enabled!


In [45]:
lines = app_code.splitlines()

start = next(
    i for i, line in enumerate(lines)
    if 'elif page == "Ultrasound Analysis":' in line
)

for i in range(start, len(lines)):
    print(f"{i+1}: {lines[i]}")

401: elif page == "Ultrasound Analysis":
402: 
403:     st.title("Ultrasound Analysis")
404: 
405:     st.write("Upload an ultrasound image for model-based classification.")
406: 
407:     uploaded_file = st.file_uploader(
408:         "Upload ultrasound image",
409:         type=["jpg", "jpeg", "png"]
410:     )
411: 
412:     if uploaded_file is not None:
413: 
414:         image = Image.open(uploaded_file).convert("L")
415: 
416:         st.image(
417:             image,
418:             caption="Uploaded Ultrasound Image",
419:             width=350
420:         )
421: 
422:         if st.button("Analyze Ultrasound"):
423: 
424:             try:
425: 
426:                 model = load_ultrasound_model()
427: 
428:                 resized_image = image.resize(
429:                     (224, 224),
430:                     Image.Resampling.BICUBIC
431:                 )
432: 
433:                 image_array = np.array(
434:                     resized_image.convert("RGB"),
435:      

In [46]:
old_code = '''                st.divider()
                st.subheader("Ultrasound AI Advisory")

                if st.button("Generate Ultrasound AI Advisory"):
                    with st.spinner("Generating ultrasound advisory..."):
                        try:
                            advisory = generate_ultrasound_advisory(
                                prediction=prediction,
                                pcos_probability=pcos_probability,
                                comparison_probability=comparison_probability
                            )

                            st.markdown(advisory)
                            st.info(ADVISORY_DISCLAIMER)

                        except Exception as e:
                            st.error(f"Ultrasound advisory error: {e}")
'''

new_code = '''                st.divider()
                st.subheader("Ultrasound AI Advisory")

                with st.spinner("Generating ultrasound advisory..."):
                    try:
                        advisory = generate_ultrasound_advisory(
                            prediction=prediction,
                            pcos_probability=pcos_probability,
                            comparison_probability=comparison_probability
                        )

                        st.markdown(advisory)
                        st.info(ADVISORY_DISCLAIMER)

                    except Exception as e:
                        st.error(f"Ultrasound advisory error: {e}")
'''

if old_code in app_code:
    app_code = app_code.replace(old_code, new_code, 1)
    print("Ultrasound advisory button removed. Automatic advisory enabled!")
else:
    print("Exact section not found. No changes made.")

Ultrasound advisory button removed. Automatic advisory enabled!


In [47]:
import ast

ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Syntax valid. Updated app.py saved successfully!")

Syntax valid. Updated app.py saved successfully!


In [48]:
!pkill -f streamlit || true

^C


In [50]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Groq API key reloaded from Colab Secrets.")

Groq API key reloaded from Colab Secrets.


In [51]:
lines = app_code.splitlines()

start = next(
    i for i, line in enumerate(lines)
    if "Patient-Specific SHAP Explanation" in line
)

for i in range(start - 12, start + 35):
    print(f"{i+1}: {lines[i]}")

346:             patient_shap = shap_values[0, :, predicted_index]
347: 
348:             shap_df = pd.DataFrame({
349:                 "Feature": FEATURES,
350:                 "SHAP Value": patient_shap
351:             })
352: 
353:             shap_df["Absolute SHAP"] = np.abs(shap_df["SHAP Value"])
354:             shap_df = shap_df.sort_values(
355:                 "Absolute SHAP", ascending=False
356:             ).head(5)
357: 
358:             st.subheader("Patient-Specific SHAP Explanation")
359:             st.caption(
360:                 "Top five features contributing to this model prediction. "
361:                 "SHAP values represent model contributions, not medical causes."
362:             )
363: 
364:             st.bar_chart(shap_df.set_index("Feature")["SHAP Value"])
365: 
366:             st.dataframe(
367:                 shap_df[["Feature", "SHAP Value"]],
368:                 hide_index=True
369:             )
370: 
371:             # 4. Automatically genera

In [52]:
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")

print("Key exists:", bool(key))
print("Key length:", len(key) if key else 0)
print("Key has surrounding spaces:", key != key.strip() if key else False)

Key exists: True
Key length: 6
Key has surrounding spaces: False


In [53]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()

print("Groq key loaded into environment.")

Groq key loaded into environment.


In [58]:
from groq import Groq
import os

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Reply with: Groq connection successful"}
    ],
    max_tokens=30
)

print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [57]:
from google.colab import userdata
import os

key = userdata.get("GROQ_API_KEY")

print("Secret exists:", bool(key))
print("Key length:", len(key) if key else 0)
print("Starts with expected prefix:", key.startswith("gsk_") if key else False)

os.environ["GROQ_API_KEY"] = key.strip() if key else ""

Secret exists: True
Key length: 56
Starts with expected prefix: True


In [59]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Reply with: Groq connection successful"}
    ],
    max_tokens=30
)

print(response.choices[0].message.content)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [60]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Reply with: Groq connection successful"}
    ],
    max_tokens=30
)

print(response.choices[0].message.content)

Groq


In [61]:
print("Response:", repr(response.choices[0].message.content))

Response: 'Groq'


In [62]:
print([line for line in app_code.splitlines() if "GROQ_MODEL" in line])

['GROQ_MODEL = "llama-3.3-70b-versatile"', '        model=GROQ_MODEL,', '        model=GROQ_MODEL,']


In [63]:
app_code = app_code.replace(
    'GROQ_MODEL = "llama-3.3-70b-versatile"',
    'GROQ_MODEL = "openai/gpt-oss-120b"'
)

print("Updated model:")
print([line for line in app_code.splitlines() if "GROQ_MODEL" in line])

Updated model:
['GROQ_MODEL = "openai/gpt-oss-120b"', '        model=GROQ_MODEL,', '        model=GROQ_MODEL,']


In [64]:
with open(app_path, "w") as file:
    file.write(app_code)

print("PCOS Insight app saved successfully!")

PCOS Insight app saved successfully!


In [65]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [97]:
!pkill -f streamlit || true

^C


In [100]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import time
import re

# Start Streamlit
streamlit = subprocess.Popen(
    [
        "streamlit", "run",
        "/content/drive/MyDrive/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Start Cloudflare Tunnel
cloudflare = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting PCOS Insight...")

for line in cloudflare.stdout:
    match = re.search(
        r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com",
        line
    )
    if match:
        print("\nPCOS Insight is ready!")
        print("Open this URL:", match.group(0))
        break

(Reading database ... 122817 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) over (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Starting PCOS Insight...


In [101]:
!pkill -f streamlit || true

^C


In [68]:
print(app_code[:5000])


import streamlit as st
import pandas as pd
import numpy as np
import xgboost as xgb
import tensorflow as tf
import os

from PIL import Image
from tensorflow.keras.applications.resnet50 import preprocess_input

# --------------------------------------------------
# GROQ GENAI CONFIGURATION
# --------------------------------------------------
from groq import Groq

GROQ_MODEL = "openai/gpt-oss-120b"

ADVISORY_DISCLAIMER = (
    "This AI-generated information is for educational purposes only. "
    "It is not a medical diagnosis or treatment recommendation. "
    "Please consult a qualified healthcare professional."
)

def get_groq_client():
    key = os.environ.get("GROQ_API_KEY")

    if not key:
        return None

    return Groq(api_key=key)


# --------------------------------------------------
# CLINICAL AI ADVISORY
# --------------------------------------------------

def generate_clinical_advisory(subtype, confidence, patient_features, shap_features):

    client = get_groq_cli

In [69]:
position = app_code.find("st.set_page_config")
print(app_code[position:position+150])

st.set_page_config(
    page_title="PCOS Insight",
    page_icon="🌸",
    layout="wide"
)

# --------------------------------------------------
# MODE


In [70]:
theme_code = """
st.markdown(\"\"\"
<style>

.stApp {
    background: linear-gradient(135deg, #F8F5FF, #F1ECFA);
}

[data-testid="stSidebar"] {
    background-color: #EAE2F7;
}

h1, h2, h3 {
    color: #604582 !important;
}

p, label {
    color: #493B5D;
}

[data-testid="stMetric"],
[data-testid="stForm"] {
    background-color: #FFFFFF;
    border: 1px solid #E5D9F3;
    border-radius: 14px;
    padding: 15px;
}

.stButton > button {
    background-color: #8064A8;
    color: white;
    border: none;
    border-radius: 10px;
}

.stButton > button:hover {
    background-color: #674B91;
    color: white;
}

[data-testid="stAlert"] {
    border-radius: 12px;
}

</style>
\"\"\", unsafe_allow_html=True)
"""

marker = 'st.set_page_config('
position = app_code.find(marker)

if position == -1:
    print("Page configuration not found. No changes made.")
else:
    end = app_code.find(')', position) + 1
    app_code = app_code[:end] + "\n\n" + theme_code + app_code[end:]
    print("Lavender theme inserted successfully!")

Lavender theme inserted successfully!


In [71]:
import ast

ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("PCOS Insight updated and saved!")

PCOS Insight updated and saved!


In [72]:
start = app_code.find('if page == "Home":')

if start == -1:
    start = app_code.find('if page == "Home"')

print(app_code[start:start+2500])

if page == "Home":

    st.title("🌸 PCOS Insight")

    st.subheader("AI-assisted PCOS Research and Analysis")

    st.write("""
    PCOS Insight is a research prototype that explores
    clinical and ultrasound-based PCOS analysis.

    **Available modules:**

    - Clinical Analysis: Four research-defined PCOS subtypes
      using clinical features and XGBoost.
    - Patient-specific SHAP: Understand the features influencing the clinical prediction.
      clinical features on model predictions.
    - Ultrasound Analysis: Ultrasound image classification
      using a CNN model.
    """)

    st.info(
        "This application is for research and educational "
        "purposes only. It is not a medical diagnostic tool."
    )


# ==================================================
# CLINICAL ANALYSIS
# ==================================================

elif page == "Clinical Analysis":

    st.title("Clinical Analysis")

    st.write("Enter the clinical measurements below.")

    with

In [73]:

start = app_code.find('if page == "Home":')
end = app_code.find('elif page == "Clinical Analysis":')

if start == -1 or end == -1:
    print("Home page section not found. No changes made.")

else:
    new_home = '''
if page == "Home":

    st.markdown("""
    <div style="
        background: linear-gradient(135deg, #E9DDF8, #F5EFFF, #E5DDF5);
        padding: 45px 35px;
        border-radius: 22px;
        text-align: center;
        margin-bottom: 25px;
    ">
        <h1 style="color:#604582; font-size:42px;">
            🌸 PCOS Insight
        </h1>
        <h3 style="color:#8064A8;">
            Understand your body. Embrace your journey.
        </h3>
        <p style="color:#594B6D; font-size:17px;">
            Every woman's health journey is unique.
            Explore your health insights with care,
            understanding and confidence.
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown(
        "<h3 style='color:#604582;'>Welcome to your health insight space 💜</h3>",
        unsafe_allow_html=True
    )

    st.write("""
    PCOS Insight is an AI-assisted research application
    designed to explore PCOS-related clinical features
    and ultrasound image patterns.
    """)

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("""
        <div style="
            background:#F0E9FA;
            padding:25px;
            border-radius:16px;
            min-height:175px;
        ">
            <h3 style="color:#604582;">🩷 Clinical Analysis</h3>
            <p style="color:#594B6D;">
                Explore clinical measurements, research-defined
                PCOS subtype predictions and patient-specific
                SHAP explanations.
            </p>
        </div>
        """, unsafe_allow_html=True)

    with col2:
        st.markdown("""
        <div style="
            background:#F7EDF8;
            padding:25px;
            border-radius:16px;
            min-height:175px;
        ">
            <h3 style="color:#604582;">🔬 Ultrasound Analysis</h3>
            <p style="color:#594B6D;">
                Explore AI-assisted ultrasound image
                classification and its prediction probabilities.
            </p>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("""
    <div style="
        background:#FFFFFF;
        padding:22px;
        border-radius:16px;
        border-left:5px solid #A78BCB;
        margin-top:25px;
    ">
        <h4 style="color:#604582;">A gentle reminder 💜</h4>
        <p style="color:#594B6D; font-style:italic;">
            "You are more than your diagnosis.
            Be kind to yourself, trust your journey,
            and celebrate every small step."
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.caption(
        "Research and educational prototype only. "
        "AI predictions are not medical diagnoses."
    )


'''

    app_code = app_code[:start] + new_home + app_code[end:]

    print("New lavender Home page inserted successfully!")


New lavender Home page inserted successfully!


In [74]:
import ast

ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Home page updated and saved successfully!")

Home page updated and saved successfully!


In [75]:
sidebar_css = """
<style>

/* Sidebar title */
[data-testid="stSidebar"] h1 {
    color: #604582 !important;
    font-size: 27px;
}

/* Sidebar background */
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #EAE2F7, #F5F0FC);
}

/* Navigation labels */
[data-testid="stSidebar"] label {
    color: #604582 !important;
    font-weight: 600;
}

/* Radio button spacing */
[data-testid="stSidebar"] [role="radiogroup"] {
    gap: 12px;
}

/* Main content spacing */
.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
}

</style>
"""

app_code = app_code.replace(
    'st.markdown("""',
    'st.markdown("""',
    0
)

marker = 'st.sidebar.title("🌸 PCOS Insight")'

if sidebar_css not in app_code:
    app_code = app_code.replace(
        marker,
        'st.markdown(' + repr(sidebar_css) +
        ', unsafe_allow_html=True)\n\n' + marker
    )

print("Sidebar styling added!")

Sidebar styling added!


In [76]:
import ast

ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("PCOS Insight saved successfully!")

PCOS Insight saved successfully!


In [79]:
start = app_code.find("# SIDEBAR NAVIGATION")
print(app_code[start:start+1500])

# SIDEBAR NAVIGATION
# --------------------------------------------------

st.markdown('\n<style>\n\n/* Sidebar title */\n[data-testid="stSidebar"] h1 {\n    color: #604582 !important;\n    font-size: 27px;\n}\n\n/* Sidebar background */\n[data-testid="stSidebar"] {\n    background: linear-gradient(180deg, #EAE2F7, #F5F0FC);\n}\n\n/* Navigation labels */\n[data-testid="stSidebar"] label {\n    color: #604582 !important;\n    font-weight: 600;\n}\n\n/* Radio button spacing */\n[data-testid="stSidebar"] [role="radiogroup"] {\n    gap: 12px;\n}\n\n/* Main content spacing */\n.block-container {\n    padding-top: 2rem;\n    padding-bottom: 2rem;\n}\n\n</style>\n', unsafe_allow_html=True)

st.sidebar.title("🌸 PCOS Insight")

page = st.sidebar.radio(
    "Navigate",
    [
        "Home",
        "Clinical Analysis",
                "Ultrasound Analysis"
    ]
)

# ==================================================
# HOME PAGE
# ==================================================


if page == "

In [80]:
for i, line in enumerate(app_code.splitlines()):
    if any(x in line for x in [
        "Predict Subtype",
        "Upload",
        "st.form_submit_button",
        "st.file_uploader"
    ]):
        print(i, line)

377         submitted = st.form_submit_button("Predict Subtype")
518     st.write("Upload an ultrasound image for model-based classification.")
520     uploaded_file = st.file_uploader(
521         "Upload ultrasound image",
531             caption="Uploaded Ultrasound Image",


In [81]:
start = app_code.find("# SIDEBAR NAVIGATION")
end = app_code.find("# HOME PAGE")

if start == -1 or end == -1:
    print("Navigation section not found. No changes made.")
else:
    new_navigation = '''
# --------------------------------------------------
# PAGE NAVIGATION — NO SIDEBAR
# --------------------------------------------------

if "page" not in st.session_state:
    st.session_state.page = "Home"

page = st.session_state.page

'''

    app_code = app_code[:start] + new_navigation + app_code[end:]
    print("Sidebar navigation removed!")

Sidebar navigation removed!


In [82]:
start = app_code.find('if page == "Home":')
end = app_code.find('elif page == "Clinical Analysis":')

if start == -1 or end == -1:
    print("Home section not found. No changes made.")
else:
    new_home = '''
if page == "Home":

    st.markdown("""
    <div style="
        background:linear-gradient(135deg,#E9DDF8,#F5EFFF,#E5DDF5);
        padding:35px 25px;
        border-radius:22px;
        text-align:center;
    ">
        <div style="font-size:35px;">🌸</div>
        <h1 style="color:#604582;">PCOS Insight</h1>
        <h3 style="color:#8064A8;">
            Understand your body. Embrace your journey.
        </h3>
        <p style="color:#594B6D;font-size:17px;">
            Every woman's health journey is unique.
            Explore your health insights with care and confidence.
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("### Welcome to your health insight space 💜")
    st.write(
        "Choose an analysis to explore your health-related "
        "clinical features or ultrasound image patterns."
    )

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("""
        <div style="background:#F0E9FA;padding:22px;
        border-radius:16px;min-height:145px;">
            <h3 style="color:#604582;">🩷 Clinical Analysis</h3>
            <p style="color:#594B6D;">
                Explore clinical measurements, subtype predictions
                and patient-specific SHAP explanations.
            </p>
        </div>
        """, unsafe_allow_html=True)

        if st.button("Explore Clinical Analysis", use_container_width=True):
            st.session_state.page = "Clinical Analysis"
            st.rerun()

    with col2:
        st.markdown("""
        <div style="background:#F7EDF8;padding:22px;
        border-radius:16px;min-height:145px;">
            <h3 style="color:#604582;">🔬 Ultrasound Analysis</h3>
            <p style="color:#594B6D;">
                Explore AI-assisted ultrasound image classification
                and prediction probabilities.
            </p>
        </div>
        """, unsafe_allow_html=True)

        if st.button("Explore Ultrasound Analysis", use_container_width=True):
            st.session_state.page = "Ultrasound Analysis"
            st.rerun()

    st.markdown("""
    <div style="background:white;padding:20px;
    border-radius:15px;border-left:5px solid #A78BCB;
    margin-top:25px;">
        <h4 style="color:#604582;">A gentle reminder 💜</h4>
        <p style="color:#594B6D;font-style:italic;">
            You are more than your diagnosis. Be kind to yourself
            and celebrate every small step.
        </p>
    </div>
    """, unsafe_allow_html=True)

    st.caption(
        "Research and educational prototype only. "
        "AI predictions are not medical diagnoses."
    )


'''

    app_code = app_code[:start] + new_home + app_code[end:]
    print("Clickable homepage navigation added!")

Clickable homepage navigation added!


In [83]:
clinical_marker = 'elif page == "Clinical Analysis":'
ultrasound_marker = 'elif page == "Ultrasound Analysis":'

for marker in [clinical_marker, ultrasound_marker]:
    position = app_code.find(marker)

    if position == -1:
        print("Could not find:", marker)
        continue

    next_line = app_code.find("\n", position) + 1

    back_button = '''
    if st.button("← Back to Home"):
        st.session_state.page = "Home"
        st.rerun()

'''

    app_code = app_code[:next_line] + back_button + app_code[next_line:]

print("Back to Home buttons added!")

Back to Home buttons added!


In [84]:
button_css = """
/* Lavender action buttons */
.stButton > button,
.stFormSubmitButton > button {
    background-color: #8064A8 !important;
    color: #FFFFFF !important;
    border: 1px solid #8064A8 !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
}

.stButton > button:hover,
.stFormSubmitButton > button:hover {
    background-color: #674B91 !important;
    color: #FFFFFF !important;
    border: 1px solid #674B91 !important;
}
"""

if button_css.strip() not in app_code:
    if "</style>" in app_code:
        app_code = app_code.replace(
            "</style>",
            button_css + "\n</style>",
            1
        )
        print("Lavender button styling added!")
    else:
        print("CSS closing tag not found. No changes made.")
else:
    print("Button styling is already present.")

Lavender button styling added!


In [85]:
import ast

ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Updated app.py saved successfully!")

Updated app.py saved successfully!


In [88]:

dark_mode_css = """
/* Improve readability in both light and dark modes */

[data-testid="stAppViewContainer"] {
    color: var(--text-color);
}

[data-testid="stMarkdownContainer"] p,
[data-testid="stMarkdownContainer"] li,
[data-testid="stWidgetLabel"] p,
label {
    color: var(--text-color) !important;
}

/* Input fields */
input, textarea, [data-baseweb="select"] {
    color: var(--text-color) !important;
}

/* Headings */
h1, h2, h3 {
    color: var(--text-color) !important;
}

/* Keep button text white */
.stButton > button,
.stFormSubmitButton > button {
    color: #FFFFFF !important;
}
"""

if "/* Improve readability in both light and dark modes */" not in app_code:
    app_code = app_code.replace(
        "</style>",
        dark_mode_css + "\n</style>",
        1
    )

import ast
ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Theme readability CSS added and app.py saved!")


Theme readability CSS added and app.py saved!


In [91]:

import re
import ast

# Remove the previously added automatic dark-mode CSS
app_code = re.sub(
    r'/\* Improve readability in both light and dark modes \*/.*?(?=</style>)',
    '',
    app_code,
    flags=re.DOTALL
)

# Add fixed light-theme styling
fixed_theme_css = """
/* Keep PCOS Insight in its lavender light theme */

[data-testid="stAppViewContainer"] {
    background: linear-gradient(135deg, #F8F5FF, #F1ECFA) !important;
}

h1, h2, h3,
[data-testid="stMarkdownContainer"] p,
[data-testid="stMarkdownContainer"] li,
label {
    color: #604582 !important;
}

/* Hide Streamlit toolbar and theme selector */
[data-testid="stToolbar"],
[data-testid="stDecoration"],
header[data-testid="stHeader"] {
    display: none !important;
}

/* Preserve white text on purple buttons */
.stButton > button,
.stFormSubmitButton > button {
    background-color: #8064A8 !important;
    color: #FFFFFF !important;
}
"""

if "/* Keep PCOS Insight in its lavender light theme */" not in app_code:
    app_code = app_code.replace(
        "</style>",
        fixed_theme_css + "\n</style>",
        1
    )

# Validate and save
ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Fixed lavender theme saved successfully!")


Fixed lavender theme saved successfully!


In [94]:

# Change all button text to black
app_code = app_code.replace(
    "color: #FFFFFF !important;",
    "color: #000000 !important;"
)

import ast
ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Button font colour changed to black!")


Button font colour changed to black!


In [95]:

for i, line in enumerate(app_code.splitlines()):
    if any(word in line.lower() for word in [
        "predicted", "st.success", "pco", "comparison class"
    ]):
        print(f"{i}: {line}")


45:     You are an educational PCOS health information assistant.
47:     Clinical model predicted subtype: {subtype}
87: def generate_ultrasound_advisory(prediction, pcos_probability,
96:     You are an educational PCOS health information assistant.
99:     PCO probability: {pcos_probability:.2%}
100:     Comparison Class probability: {comparison_probability:.2%}
110:     Do not diagnose PCOS or infer clinical subtypes or hormone levels.
132:     page_title="PCOS Insight",
201: /* Keep PCOS Insight in its lavender light theme */
236: CLINICAL_MODEL_PATH = "/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"
239:     "/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras"
299:         <h1 style="color:#604582;">PCOS Insight</h1>
449:             predicted_index = int(np.argmax(probabilities))
453:             # 1. Display predicted subtype first
454:             st.success(
455:                 f"Predicted PCOS Subtype: {CLASSES[predicted_index]}"
460:  

In [96]:

# 1. Clinical prediction: plain red text
old_clinical = '''            st.success(
                f"Predicted PCOS Subtype: {CLASSES[predicted_index]}"
            )'''

new_clinical = '''            st.markdown(
                f"<h3 style='color:#D32F2F;'>Predicted PCOS Subtype: {CLASSES[predicted_index]}</h3>",
                unsafe_allow_html=True
            )'''

# 2. Ultrasound prediction: red for PCO, green for Comparison Class
old_ultrasound = '''                st.success(f"Predicted class: {prediction}")'''

new_ultrasound = '''                prediction_color = "#D32F2F" if prediction == "PCO" else "#218838"

                st.markdown(
                    f"<h3 style='color:{prediction_color};'>Predicted class: {prediction}</h3>",
                    unsafe_allow_html=True
                )'''

if old_clinical in app_code:
    app_code = app_code.replace(old_clinical, new_clinical, 1)
    print("Clinical prediction display updated!")
else:
    print("Clinical display code not found exactly.")

if old_ultrasound in app_code:
    app_code = app_code.replace(old_ultrasound, new_ultrasound, 1)
    print("Ultrasound prediction display updated!")
else:
    print("Ultrasound display code not found exactly.")

# Validate and save
import ast
ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Updated app.py saved successfully!")


Clinical prediction display updated!
Ultrasound prediction display updated!
Updated app.py saved successfully!


In [102]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import time
import re

# Start Streamlit
streamlit = subprocess.Popen(
    [
        "streamlit", "run",
        "/content/drive/MyDrive/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Start Cloudflare Tunnel
cloudflare = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting PCOS Insight...")

for line in cloudflare.stdout:
    match = re.search(
        r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com",
        line
    )
    if match:
        print("\nPCOS Insight is ready!")
        print("Open this URL:", match.group(0))
        break

(Reading database ... 122817 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) over (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Starting PCOS Insight...

PCOS Insight is ready!
Open this URL: https://arms-prominent-catalyst-authors.trycloudflare.com


In [103]:
import subprocess
import requests
import time

print("Checking Streamlit...")

try:
    r = requests.get("http://127.0.0.1:8501", timeout=10)
    print("Streamlit status:", r.status_code)
except Exception as e:
    print("Streamlit error:", e)

print("\nChecking Cloudflare process...")

result = subprocess.run(
    ["pgrep", "-af", "cloudflared"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "Cloudflare process not found")

Checking Streamlit...
Streamlit status: 200

Checking Cloudflare process...
8880 cloudflared tunnel --url http://localhost:8501
11683 cloudflared tunnel --url http://localhost:8501
12899 cloudflared tunnel --url http://localhost:8501
16615 cloudflared tunnel --url http://localhost:8501
34297 cloudflared tunnel --url http://localhost:8501
44345 cloudflared tunnel --url http://localhost:8501
47741 cloudflared tunnel --url http://localhost:8501
51846 cloudflared tunnel --url http://localhost:8501
53534 cloudflared tunnel --url http://localhost:8501
55925 cloudflared tunnel --url http://localhost:8501
57479 cloudflared tunnel --url http://localhost:8501



In [104]:
!pkill -f cloudflared || true

^C


In [105]:
!pgrep -af cloudflared

8880 cloudflared tunnel --url http://localhost:8501
16615 cloudflared tunnel --url http://localhost:8501
34297 cloudflared tunnel --url http://localhost:8501
44345 cloudflared tunnel --url http://localhost:8501
47741 cloudflared tunnel --url http://localhost:8501
51846 cloudflared tunnel --url http://localhost:8501
53534 cloudflared tunnel --url http://localhost:8501
57479 [cloudflared] <defunct>


In [106]:
import subprocess

result = subprocess.run(
    ["bash", "-c",
     "pids=$(pgrep -x cloudflared); "
     "if [ -n \"$pids\" ]; then kill -9 $pids; fi"],
    capture_output=True,
    text=True
)

print("Cloudflare cleanup attempted.")

Cloudflare cleanup attempted.


In [107]:
import subprocess

result = subprocess.run(
    ["pgrep", "-af", "cloudflared"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "All Cloudflare tunnels stopped!")

57479 [cloudflared] <defunct>



In [113]:
import subprocess
import time
import re

# Start one fresh tunnel
cloudflare = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting a fresh Cloudflare tunnel...")

for line in cloudflare.stdout:
    print(line.strip())

    match = re.search(
        r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com",
        line
    )

    if match:
        print("\nYour new PCOS Insight public link:")
        print(match.group(0))
        break

Starting a fresh Cloudflare tunnel...
2026-11-23T15:17:56Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-23T15:17:56Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-23T15:17:59Z INF +--------------------------------------------------------------------------------------------+
2026-11-23T15:17:59Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-23T15:17:59Z INF |  https://merit-t

In [109]:

# Fix advisory readability and prediction colours
updated_css = """
/* Make normal app text readable */
[data-testid="stMarkdownContainer"],
[data-testid="stMarkdownContainer"] p,
[data-testid="stMarkdownContainer"] li {
    color: #493B5D !important;
}

/* Clinical prediction - red */
.prediction-red {
    color: #D32F2F !important;
    font-size: 23px;
    font-weight: 700;
}

/* Ultrasound prediction - red or green */
.prediction-green {
    color: #218838 !important;
    font-size: 23px;
    font-weight: 700;
}

/* Advisory content */
.advisory-text {
    color: #493B5D !important;
    font-size: 16px;
}
"""

if "/* Make normal app text readable */" not in app_code:
    app_code = app_code.replace(
        "</style>",
        updated_css + "\n</style>",
        1
    )

import ast
ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("Text styling updated and saved!")


Text styling updated and saved!


In [110]:

lines = app_code.splitlines()

for start, end in [(445, 465), (510, 540), (600, 650)]:
    print(f"\n--- Lines {start} to {end} ---")
    for i in range(start, min(end, len(lines))):
        print(f"{i}: {lines[i]}")



--- Lines 445 to 465 ---
445:             "AMH": amh,
446:             "Androgen_Score": androgen,
447:             "HOMA_IR": homa,
448:             "FSH_LH_raw": fsh_lh_raw,
449:             "Total_Follicles": total_follicles,
450:             "Follicle_L": follicle_l,
451:             "LH": lh,
452:             "FSH": fsh,
453:             "LH_FSH_Ratio": lh_fsh_ratio,
454:             "Cycle_Length": cycle_length,
455:             "Fast_Food": 1 if fast_food == "Yes" else 0,
456:             "TSH": tsh,
457:             "BP_Diastolic": bp,
458:             "BMI": bmi,
459:             "Progesterone": progesterone,
460:             "Marriage_Years": marriage_years,
461:             "RR": rr
462:         }
463: 
464:         try:

--- Lines 510 to 540 ---
510:                 use_container_width=True,
511:                 hide_index=True
512:             )
513: 
514:             # 3. Patient-specific SHAP explanation
515:             import shap
516: 
517:             explainer = sh

In [111]:

lines = app_code.splitlines()

for start, end in [(475, 510), (535, 590), (635, 690)]:
    print(f"\n--- Lines {start} to {end} ---")
    for i in range(start, min(end, len(lines))):
        print(f"{i}: {lines[i]}")



--- Lines 475 to 510 ---
475:             probabilities = probabilities[0]
476: 
477:             predicted_index = int(np.argmax(probabilities))
478: 
479: 
480: 
481:             # 1. Display predicted subtype first
482:             st.markdown(
483:                 f"<h3 style='color:#D32F2F;'>Predicted PCOS Subtype: {CLASSES[predicted_index]}</h3>",
484:                 unsafe_allow_html=True
485:             )
486: 
487:             st.metric(
488:                 "Prediction Confidence",
489:                 f"{probabilities[predicted_index] * 100:.2f}%"
490:             )
491: 
492:             # 2. Probability distribution
493:             probability_df = pd.DataFrame({
494:                 "Subtype": CLASSES,
495:                 "Probability": probabilities
496:             })
497: 
498:             probability_df["Probability (%)"] = (
499:                 probability_df["Probability"] * 100
500:             ).round(2)
501: 
502:             st.subheader("Subtype Probabili

In [112]:

# 1. Fix clinical prediction display
old_clinical = '''            st.markdown(
                f"<h3 style='color:#D32F2F;'>Predicted PCOS Subtype: {CLASSES[predicted_index]}</h3>",
                unsafe_allow_html=True
            )'''

new_clinical = '''            st.markdown(
                f"<div style='color:#D32F2F !important; font-size:23px; font-weight:700;'>Predicted PCOS Subtype: {CLASSES[predicted_index]}</div>",
                unsafe_allow_html=True
            )'''

# 2. Fix ultrasound prediction display
old_ultrasound = '''                st.markdown(
                    f"<h3 style='color:{prediction_color};'>Predicted class: {prediction}</h3>",
                    unsafe_allow_html=True
                )'''

new_ultrasound = '''                st.markdown(
                    f"<div style='color:{prediction_color} !important; font-size:23px; font-weight:700;'>Predicted class: {prediction}</div>",
                    unsafe_allow_html=True
                )'''

if old_clinical in app_code:
    app_code = app_code.replace(old_clinical, new_clinical, 1)
    print("Clinical prediction styling fixed.")
else:
    print("Clinical prediction code was not found.")

if old_ultrasound in app_code:
    app_code = app_code.replace(old_ultrasound, new_ultrasound, 1)
    print("Ultrasound prediction styling fixed.")
else:
    print("Ultrasound prediction code was not found.")

# 3. Add strong readable styling for GenAI advisory content
advisory_css = """
/* GenAI advisory readable text */
.stMarkdown p,
.stMarkdown li,
.stMarkdown strong,
.stMarkdown em,
.stMarkdown h1,
.stMarkdown h2,
.stMarkdown h3 {
    color: #493B5D !important;
}
"""

if "/* GenAI advisory readable text */" not in app_code:
    app_code = app_code.replace(
        "</style>",
        advisory_css + "\n</style>",
        1
    )

# Validate and save
import ast
ast.parse(app_code)

with open(app_path, "w") as file:
    file.write(app_code)

print("All display styling changes saved successfully!")


Clinical prediction styling fixed.
Ultrasound prediction styling fixed.
All display styling changes saved successfully!


In [114]:
import os
import shutil

# Create deployment folder
deploy_dir = "/content/pcos-insight"

os.makedirs(deploy_dir, exist_ok=True)
os.makedirs(os.path.join(deploy_dir, "models"), exist_ok=True)

# Copy app.py
shutil.copy(
    "/content/drive/MyDrive/app.py",
    os.path.join(deploy_dir, "app.py")
)

# Copy trained models
model_source = "/content/drive/MyDrive/pcos_project/outputs/models"

shutil.copy(
    os.path.join(model_source, "xgboost_model.json"),
    os.path.join(deploy_dir, "models", "xgboost_model.json")
)

shutil.copy(
    os.path.join(model_source, "cnn_model.keras"),
    os.path.join(deploy_dir, "models", "cnn_model.keras")
)

print("Deployment folder created!")
print(os.listdir(deploy_dir))
print("Models:", os.listdir(os.path.join(deploy_dir, "models")))

Deployment folder created!
['models', 'app.py']
Models: ['cnn_model.keras', 'xgboost_model.json']


In [115]:
deploy_app_path = "/content/pcos-insight/app.py"

with open(deploy_app_path, "r") as file:
    deploy_code = file.read()

deploy_code = deploy_code.replace(
    '"/content/drive/MyDrive/pcos_project/outputs/models/xgboost_model.json"',
    '"models/xgboost_model.json"'
)

deploy_code = deploy_code.replace(
    '"/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras"',
    '"models/cnn_model.keras"'
)

with open(deploy_app_path, "w") as file:
    file.write(deploy_code)

print("Deployment model paths updated!")

Deployment model paths updated!


In [116]:
for line in deploy_code.splitlines():
    if "MODEL_PATH" in line:
        print(line)

CLINICAL_MODEL_PATH = "models/xgboost_model.json"
ULTRASOUND_MODEL_PATH = (
    model.load_model(CLINICAL_MODEL_PATH)
    return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)


In [117]:
requirements = """streamlit
pandas
numpy
matplotlib
seaborn
scikit-learn
xgboost
shap
tensorflow
Pillow
groq
"""

with open("/content/pcos-insight/requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created!")

requirements.txt created!


In [118]:
import shutil

shutil.make_archive(
    "/content/pcos-insight",
    "zip",
    "/content/pcos-insight"
)

from google.colab import files
files.download("/content/pcos-insight.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [119]:
import os

cnn_path = "/content/pcos-insight/models/cnn_model.keras"

size_mb = os.path.getsize(cnn_path) / (1024 * 1024)

print(f"CNN model size: {size_mb:.2f} MB")

CNN model size: 90.60 MB


In [120]:
!pip install -q huggingface_hub

In [121]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)

print("Hugging Face connected successfully!")

KeyboardInterrupt: Interrupted by user

In [122]:
from huggingface_hub import HfApi
from getpass import getpass

hf_token = getpass("Enter your Hugging Face token: ")
api = HfApi()
api.whoami(token=hf_token)

print("Token verified successfully!")

Enter your Hugging Face token: ··········
Token verified successfully!


In [123]:
from huggingface_hub import HfApi

api = HfApi()

model_path = "/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras"

api.upload_file(
    path_or_fileobj=model_path,
    path_in_repo="cnn_model.keras",
    repo_id="MahaHuggy/pcos-insight-models",
    repo_type="model",
    commit_message="Upload PCOS ultrasound CNN model"
)

print("CNN model uploaded successfully!")

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6b046574-733e710f2d6bccfb1f59064c;7b3cc839-fe77-43d1-ba06-7b7f2c8379de)

Repository Not Found for url: https://huggingface.co/api/models/MahaHuggy/pcos-insight-models/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [124]:
from huggingface_hub import HfApi

api = HfApi()

try:
    user = api.whoami()
    print("Logged in as:", user["name"])
except Exception as e:
    print("Authentication failed:", e)

Authentication failed: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.


In [125]:
from huggingface_hub import login, HfApi
from getpass import getpass

hf_token = getpass("Paste your Hugging Face WRITE token: ")

login(token=hf_token, add_to_git_credential=False)

api = HfApi(token=hf_token)

user = api.whoami()
print("Logged in as:", user["name"])

Paste your Hugging Face WRITE token: ··········
Logged in as: MahaHuggy


In [126]:
api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/pcos_project/outputs/models/cnn_model.keras",
    path_in_repo="cnn_model.keras",
    repo_id="MahaHuggy/pcos-insight-models",
    repo_type="model",
    commit_message="Upload PCOS ultrasound CNN model"
)

print("CNN model uploaded successfully!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ts/models/cnn_model.keras:   1%|          |  558kB / 95.0MB            

Validating                    :           |   0%            

CNN model uploaded successfully!


In [127]:
deploy_app_path = "/content/pcos-insight/app.py"

with open(deploy_app_path, "r") as file:
    deploy_code = file.read()

for i, line in enumerate(deploy_code.splitlines(), 1):
    if "ULTRASOUND_MODEL_PATH" in line or "load_ultrasound" in line:
        print(i, line)

279 ULTRASOUND_MODEL_PATH = (
310 def load_ultrasound_model():
311     return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)
618                 model = load_ultrasound_model()


In [128]:
with open(deploy_app_path, "r") as file:
    lines = file.readlines()

print("".join(lines[270:315]))



# --------------------------------------------------
# MODEL AND DATA PATHS
# --------------------------------------------------

CLINICAL_MODEL_PATH = "models/xgboost_model.json"

ULTRASOUND_MODEL_PATH = (
    "models/cnn_model.keras"
)

FEATURES = [
    "Hair_Growth", "Blood_Sugar", "Hormonal_Disruption", "AMH",
    "Androgen_Score", "HOMA_IR", "FSH_LH_raw", "Total_Follicles",
    "Follicle_L", "LH", "FSH", "LH_FSH_Ratio", "Cycle_Length",
    "Fast_Food", "TSH", "BP_Diastolic", "BMI", "Progesterone",
    "Marriage_Years", "RR"
]

CLASSES = [
    "Insulin Resistant",
    "Inflammatory",
    "Adrenal",
    "Post-Pill"
]

# --------------------------------------------------
# LOAD MODELS
# --------------------------------------------------

@st.cache_resource
def load_clinical_model():
    model = xgb.Booster()
    model.load_model(CLINICAL_MODEL_PATH)
    return model


@st.cache_resource
def load_ultrasound_model():
    return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)


# --

In [129]:
deploy_app_path = "/content/pcos-insight/app.py"

with open(deploy_app_path, "r") as file:
    deploy_code = file.read()

deploy_code = deploy_code.replace(
    '''ULTRASOUND_MODEL_PATH = (
    "models/cnn_model.keras"
)''',
    '''ULTRASOUND_MODEL_PATH = "/tmp/cnn_model.keras"'''
)

deploy_code = deploy_code.replace(
    '''@st.cache_resource
def load_ultrasound_model():
    return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)''',
    '''@st.cache_resource
def load_ultrasound_model():
    from huggingface_hub import hf_hub_download

    model_path = hf_hub_download(
        repo_id="MahaHuggy/pcos-insight-models",
        filename="cnn_model.keras"
    )

    return tf.keras.models.load_model(model_path)'''
)

with open(deploy_app_path, "w") as file:
    file.write(deploy_code)

print("Ultrasound model loading updated!")

Ultrasound model loading updated!


In [130]:
deploy_app_path = "/content/pcos-insight/app.py"

with open(deploy_app_path, "r") as file:
    deploy_code = file.read()

deploy_code = deploy_code.replace(
    '''ULTRASOUND_MODEL_PATH = (
    "models/cnn_model.keras"
)''',
    '''ULTRASOUND_MODEL_PATH = "/tmp/cnn_model.keras"'''
)

deploy_code = deploy_code.replace(
    '''@st.cache_resource
def load_ultrasound_model():
    return tf.keras.models.load_model(ULTRASOUND_MODEL_PATH)''',
    '''@st.cache_resource
def load_ultrasound_model():
    from huggingface_hub import hf_hub_download

    model_path = hf_hub_download(
        repo_id="MahaHuggy/pcos-insight-models",
        filename="cnn_model.keras"
    )

    return tf.keras.models.load_model(model_path)'''
)

with open(deploy_app_path, "w") as file:
    file.write(deploy_code)

print("Ultrasound model loading updated!")

Ultrasound model loading updated!


In [131]:
with open(deploy_app_path, "r") as file:
    code = file.read()

print("Hugging Face repo configured:",
      "MahaHuggy/pcos-insight-models" in code)

print("CNN download configured:",
      "hf_hub_download" in code)

with open(requirements_path, "r") as file:
    req = file.read()

print("Hugging Face dependency added:",
      "huggingface_hub" in req)

Hugging Face repo configured: True
CNN download configured: True


NameError: name 'requirements_path' is not defined

In [132]:
requirements_path = "/content/pcos-insight/requirements.txt"

with open(requirements_path, "r") as file:
    req = file.read()

print("Hugging Face dependency added:",
      "huggingface_hub" in req)

Hugging Face dependency added: False


In [133]:
requirements_path = "/content/pcos-insight/requirements.txt"

with open(requirements_path, "a") as file:
    file.write("\nhuggingface_hub\n")

with open(requirements_path, "r") as file:
    print(file.read())

streamlit
pandas
numpy
matplotlib
seaborn
scikit-learn
xgboost
shap
tensorflow
Pillow
groq

huggingface_hub



In [134]:
with open("/content/pcos-insight/app.py", "r") as file:
    code = file.read()

print("Hugging Face code present:",
      "MahaHuggy/pcos-insight-models" in code)

print("Download function present:",
      "hf_hub_download" in code)

Hugging Face code present: True
Download function present: True


In [135]:
with open("/content/pcos-insight/requirements.txt", "r") as file:
    print(file.read())

streamlit
pandas
numpy
matplotlib
seaborn
scikit-learn
xgboost
shap
tensorflow
Pillow
groq

huggingface_hub



In [136]:
import shutil

zip_path = shutil.make_archive(
    "/content/pcos-insight-updated",
    "zip",
    "/content/pcos-insight"
)

print("Updated ZIP created:", zip_path)

Updated ZIP created: /content/pcos-insight-updated.zip


In [138]:
from google.colab import files
files.download("/content/pcos-insight-updated.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>